# Skill → ESCO Taxonomy Matching

Matches your skills (label only) against the full ESCO taxonomy using sentence embeddings.

**Strategy:**
- Query side: your skill label (short)
- Candidate side: ESCO enriched text (preferredLabel + altLabels + description)
- Top-1 match above threshold → assigned; below → flagged as unmatched
- Output: enriched skills file ready for the main pipeline

## 0. Config

In [19]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ── PATHS ───────────────────────────────────────────────────────────────────
EXCEL_PATH      = "data/Professional_growth_exercise.xlsx"  # your skills are here
ESCO_CSV_PATH   = "data/skills_en.csv"                      # ESCO taxonomy dump
CACHE_DIR       = "./embedding_cache"
OUTPUT_PATH     = "data/skills_esco_matched.csv"            # output for main pipeline
os.makedirs(CACHE_DIR, exist_ok=True)

# ── MODEL ───────────────────────────────────────────────────────────────────
# multilingual handles Italian course names later in the pipeline
MODEL_NAME = "all-mpnet-base-v2"
MATCH_THRESHOLD = 0.45

print("Config OK")

Config OK


## 1. Load Data

In [20]:
# Your skills from the Excel
df_skills = pd.read_excel(EXCEL_PATH, sheet_name="Skills")  # SkillUri, Skill
df_skills = df_skills.dropna(subset=["Skill"]).drop_duplicates(subset="SkillUri").reset_index(drop=True)
print(f"Your skills: {len(df_skills)}")
df_skills.head(3)

Your skills: 15037


,SkillUri,Skill
0,2.A.1.b,Active Listening
1,2.C.3.b,Engineering and Technology
2,2.C.4.a,Mathematics


In [21]:
# ESCO taxonomy
df_esco = pd.read_csv(ESCO_CSV_PATH, low_memory=False)

# Keep only skill/competence rows (drop occupations etc. if mixed)
if "conceptType" in df_esco.columns:
    df_esco = df_esco[df_esco["conceptType"] == "KnowledgeSkillCompetence"]

df_esco = df_esco.reset_index(drop=True)
print(f"ESCO skills: {len(df_esco)}")
df_esco[["conceptUri", "preferredLabel", "description"]].head(3)

ESCO skills: 13960


,conceptUri,preferredLabel,description
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,Assign and manage staff tasks in areas such as...
1,http://data.europa.eu/esco/skill/00064735-8fad...,supervise correctional procedures,Supervise the operations of a correctional fac...
2,http://data.europa.eu/esco/skill/000709ed-2be5...,apply anti-oppressive practices,"Identify oppression in societies, economies, c..."


## 2. Build Enriched ESCO Text

Richer candidate text = better matching against short queries.

In [22]:
def build_esco_text(row):
    parts = []

    if pd.notna(row.get("preferredLabel")):
        parts.append(str(row["preferredLabel"]))

    # altLabels are newline-separated in the ESCO dump
    if pd.notna(row.get("altLabels")) and str(row["altLabels"]).strip():
        alts = str(row["altLabels"]).replace("\n", ", ").strip()
        parts.append(alts)

    if pd.notna(row.get("description")) and str(row["description"]).strip():
        parts.append(str(row["description"]))

    if pd.notna(row.get("definition")) and str(row["definition"]).strip():
        parts.append(str(row["definition"]))

    return ". ".join(parts)

df_esco["esco_text"] = df_esco.apply(build_esco_text, axis=1)

# Sanity check
print(df_esco["esco_text"].iloc[0])

manage musical staff. manage music staff, coordinate duties of musical staff, direct musical staff, manage staff of music. Assign and manage staff tasks in areas such as scoring, arranging, copying music and vocal coaching.


## 3. Embed (with cache)

In [ ]:
import os

model = SentenceTransformer(MODEL_NAME)
print(f"Loaded: {MODEL_NAME}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22488.66it/s]


Loaded: all-mpnet-base-v2


In [24]:
def embed_with_cache(texts, cache_path, model, batch_size=64):
    if os.path.exists(cache_path):
        print(f"  cache hit: {cache_path}")
        return np.load(cache_path)
    print(f"  embedding {len(texts)} texts...")
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True   # cosine sim = dot product
    )
    np.save(cache_path, emb)
    print(f"  saved: {cache_path}")
    return emb

In [25]:
print("Embedding your skills (query side)...")
emb_your_skills = embed_with_cache(
    df_skills["Skill"].tolist(),
    os.path.join(CACHE_DIR, "emb_your_skills.npy"),
    model
)

print("Embedding ESCO enriched text (candidate side)...")
emb_esco = embed_with_cache(
    df_esco["esco_text"].tolist(),
    os.path.join(CACHE_DIR, "emb_esco_skills.npy"),
    model
)

print(f"\nShapes → yours: {emb_your_skills.shape}, ESCO: {emb_esco.shape}")

Embedding your skills (query side)...
  cache hit: ./embedding_cache/emb_your_skills.npy
Embedding ESCO enriched text (candidate side)...
  cache hit: ./embedding_cache/emb_esco_skills.npy

Shapes → yours: (15037, 384), ESCO: (13960, 384)


## 4. Match

For each of your skills, find the single best ESCO match.
Flag anything below the threshold as unmatched.

In [26]:
# Full similarity matrix: (n_your_skills, n_esco_skills)
# If ESCO is very large (13k+ skills) this is ~500MB for float32 — fine for a notebook
# If it causes memory issues, use batched dot product (see comment below)
sim_matrix = emb_your_skills @ emb_esco.T

print(f"Similarity matrix: {sim_matrix.shape}")

# ── BATCHED ALTERNATIVE (uncomment if memory is tight) ───────────────────
# top1_scores = []
# top1_indices = []
# BATCH = 256
# for i in range(0, len(emb_your_skills), BATCH):
#     batch_sim = emb_your_skills[i:i+BATCH] @ emb_esco.T
#     top1_indices.extend(batch_sim.argmax(axis=1).tolist())
#     top1_scores.extend(batch_sim.max(axis=1).tolist())

Similarity matrix: (15037, 13960)


In [27]:
top1_indices = sim_matrix.argmax(axis=1)
top1_scores  = sim_matrix.max(axis=1)

# Build result dataframe
results = []
for i, (skill_row) in df_skills.iterrows():
    score = float(top1_scores[i])
    matched = score >= MATCH_THRESHOLD
    esco_row = df_esco.iloc[top1_indices[i]]

    results.append({
        "SkillUri":           skill_row["SkillUri"],
        "Skill":              skill_row["Skill"],
        "match_score":        round(score, 4),
        "matched":            matched,
        # ESCO fields
        "esco_uri":           esco_row["conceptUri"]   if matched else None,
        "esco_label":         esco_row["preferredLabel"] if matched else None,
        "esco_altLabels":     esco_row.get("altLabels", None) if matched else None,
        "esco_description":   esco_row.get("description", None) if matched else None,
        # Enriched text for downstream embedding (main pipeline)
        "skill_text_enriched": esco_row["esco_text"] if matched else skill_row["Skill"],
    })

df_result = pd.DataFrame(results)

n_matched   = df_result["matched"].sum()
n_unmatched = (~df_result["matched"]).sum()
print(f"Matched:   {n_matched}/{len(df_result)}")
print(f"Unmatched: {n_unmatched}/{len(df_result)}")

Matched:   14991/15037
Unmatched: 46/15037


## 5. Tune the Threshold

Run this to see how match rate changes with threshold before committing.

In [28]:
print(f"{'Threshold':>10} | {'Matched':>8} | {'Unmatched':>10} | {'Match %':>8}")
print("-" * 48)
for t in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    n = (top1_scores >= t).sum()
    print(f"  {t:.2f}     | {n:>8} | {len(df_skills)-n:>10} | {n/len(df_skills)*100:>7.1f}%")

 Threshold |  Matched |  Unmatched |  Match %
------------------------------------------------
  0.30     |    15037 |          0 |   100.0%
  0.35     |    15037 |          0 |   100.0%
  0.40     |    15030 |          7 |   100.0%
  0.45     |    14991 |         46 |    99.7%
  0.50     |    14725 |        312 |    97.9%
  0.55     |    13659 |       1378 |    90.8%
  0.60     |    10826 |       4211 |    72.0%
  0.65     |     6717 |       8320 |    44.7%
  0.70     |     3064 |      11973 |    20.4%


In [29]:
# Look at some matches around the threshold to gut-check quality
borderline = df_result[
    (df_result["match_score"] >= MATCH_THRESHOLD - 0.05) &
    (df_result["match_score"] <= MATCH_THRESHOLD + 0.05)
].sort_values("match_score")

print(f"Borderline matches (score within ±0.05 of threshold {MATCH_THRESHOLD}):")
for _, row in borderline.head(10).iterrows():
    print(f"  {row['match_score']:.3f}  '{row['Skill']}'  →  '{row['esco_label']}'")

Borderline matches (score within ±0.05 of threshold 0.45):
  0.418  'Install casings to prevent cave-ins.'  →  'nan'
  0.418  'Pay small claims.'  →  'nan'
  0.421  'Press console buttons to deflect packages to predetermined accumulators or reject lines.'  →  'nan'
  0.421  'Record height or weight of patients.'  →  'nan'
  0.421  'File metal reeds until their pitches correspond with standard tuning bar pitches.'  →  'nan'
  0.424  'Level bedplate and establish centerline, using straightedge, levels, and transit.'  →  'nan'
  0.424  'Obtain customers' signatures on receipts when winnings exceed the amount held in a slot machine.'  →  'nan'
  0.426  'Fit boot spoilers, side skirts, or mud flaps to cars.'  →  'nan'
  0.427  'Grant divorces and divide assets between spouses.'  →  'nan'
  0.429  'Negotiate credit extensions when necessary.'  →  'nan'


In [30]:
# Look at unmatched skills -- are they genuinely unmatchable or is threshold too strict?
unmatched = df_result[~df_result["matched"]].copy()
# Show their best score anyway
unmatched["best_esco_candidate"] = [
    df_esco.iloc[top1_indices[i]]["preferredLabel"]
    for i in unmatched.index
]

print(f"Unmatched skills (showing best rejected candidate):")
print(unmatched[["Skill", "match_score", "best_esco_candidate"]].to_string(index=False))

Unmatched skills (showing best rejected candidate):
                                                                                                                                Skill  match_score                              best_esco_candidate
                                     Obtain customers' signatures on receipts when winnings exceed the amount held in a slot machine.       0.4241                                 process payments
                                                                                 Issue room keys and escort instructions to bellhops.       0.4488                                    allocate keys
                                                                                               Turn or reposition bedridden patients.       0.3861         follow-up on healthcare users' treatment
                                                                                      Check identifications to verify age of players.       0.4392                  

## 6. Export

The `skill_text_enriched` column is what the main pipeline should embed instead of the raw `Skill` label.
Unmatched skills fall back to their original label.

In [31]:
df_result.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"\nColumns: {df_result.columns.tolist()}")
df_result.head(3)

Saved to data/skills_esco_matched.csv

Columns: ['SkillUri', 'Skill', 'match_score', 'matched', 'esco_uri', 'esco_label', 'esco_altLabels', 'esco_description', 'skill_text_enriched']


,SkillUri,Skill,match_score,matched,esco_uri,esco_label,esco_altLabels,esco_description,skill_text_enriched
0,2.A.1.b,Active Listening,0.7565,True,http://data.europa.eu/esco/skill/a17286c5-238d...,listen actively,maintain active listening\nactive listening\na...,"Give attention to what other people say, patie...","listen actively. maintain active listening, ac..."
1,2.C.3.b,Engineering and Technology,0.5937,True,http://data.europa.eu/esco/skill/14764a75-3104...,telecommunications engineering,telecoms engineering\ntele-communications engi...,Discipline that combines computer science with...,telecommunications engineering. telecoms engin...
2,2.C.4.a,Mathematics,0.6605,True,http://data.europa.eu/esco/skill/4339176e-3acd...,mathematics,numeracy\nquantitative data\nsolid modeling\nm...,Mathematics is the study of topics such as qua...,"mathematics. numeracy, quantitative data, soli..."
